# Phase C ACT：增强训练与 episode-heldout 选点

这份 notebook **不重新录制数据，也不驱动机械臂**。它按当前诊断计划，用现有 30 条 sanitized episodes 在本机 RTX 4060 上从头训练 ACT：24 条训练、6 条按物理 cell 留出验证，在线图像增强，每 5k 保存到 80k。

验证集每个 `front/middle/back × deep/shallow` cell 各一条，并交错覆盖五个 offset。增强是保守的：每张图随机选择一种轻量亮度/对比度/饱和度/锐度、微小仿射、随机遮挡或 identity；不改变 observation/action 合同。

checkpoint 排名以 episode-balanced 的 h1 机械臂误差和闭爪阶段 h1 gripper 误差为主，同时报告 h1/h5/h20。它只能做离线筛选，不能代替闭环机械臂测试。

In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

WORKSPACE = Path.cwd()
BASE = WORKSPACE / 'training/phase_c_act_augmented'
DATASET_ROOT = WORKSPACE / 'datasets/hand_tracking_pv_carton_phase_b'
TRAIN_DATASET_ROOT = WORKSPACE / 'datasets/hand_tracking_pv_carton_phase_b_train24'
TRAIN_REPO_ID = 'local/hand_tracking_pv_carton_phase_b_train24'
SOURCE_MAP = DATASET_ROOT / 'source_episode_map.csv'
VENV_PYTHON = WORKSPACE / '.venv-lerobot/bin/python'
TRAIN_CLI = WORKSPACE / '.venv-lerobot/bin/lerobot-train'
EVALUATOR = BASE / 'evaluate_checkpoints.py'
HF_HOME = BASE / '.cache/huggingface'
MPLCONFIGDIR = BASE / '.cache/matplotlib'

# One episode from every position x depth cell, staggered across offsets.
VAL_EPISODES = [0, 5, 10, 16, 23, 26]
ALL_EPISODES = list(range(30))
TRAIN_EPISODES = sorted(set(ALL_EPISODES) - set(VAL_EPISODES))

with SOURCE_MAP.open(newline='') as handle:
    source_rows = list(csv.DictReader(handle))
heldout_rows = [row for row in source_rows if int(row['episode_index']) in VAL_EPISODES]
heldout_cells = {(row['table_position'], row['grasp_depth']) for row in heldout_rows}
expected_cells = {(position, depth) for position in ('front', 'middle', 'back') for depth in ('deep', 'shallow')}
assert len(TRAIN_EPISODES) == 24 and len(heldout_rows) == 6
assert (TRAIN_DATASET_ROOT / 'meta/info.json').is_file()
assert heldout_cells == expected_cells
assert {int(row['jaw_offset_mm']) for row in heldout_rows} == {-6, -3, 0, 3, 6}

RUN_NAME = f"act_phase_c_aug_holdout_{datetime.now():%Y%m%d_%H%M%S}"
ARTIFACT_DIR = BASE / 'runs' / RUN_NAME
SMOKE_RUN_DIR = BASE / 'outputs' / f'{RUN_NAME}_smoke'
FULL_RUN_DIR = BASE / 'outputs' / RUN_NAME
for path in (ARTIFACT_DIR, HF_HOME / 'datasets', MPLCONFIGDIR, BASE / 'outputs'):
    path.mkdir(parents=True, exist_ok=True)

print('run:', RUN_NAME)
print('train episodes:', TRAIN_EPISODES)
print('held-out episodes:')
for row in heldout_rows:
    print({key: row[key] for key in ('episode_index', 'source_episode_index', 'table_position', 'grasp_depth', 'repeat_index', 'jaw_offset_mm', 'outcome')})


## 1. 硬件 gate

必须看到本机 RTX 4060 和 CUDA；否则停止，不会偷偷退回 CPU。

In [ ]:
def clean_env():
    env = os.environ.copy()
    env.pop('PYTHONPATH', None)
    env['HF_HOME'] = str(HF_HOME)
    env['HF_DATASETS_CACHE'] = str(HF_HOME / 'datasets')
    env['MPLCONFIGDIR'] = str(MPLCONFIGDIR)
    env['PYTHONUNBUFFERED'] = '1'
    return env

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)
probe = subprocess.run(
    [str(VENV_PYTHON), '-c', "import torch; assert torch.cuda.is_available(); n=torch.cuda.get_device_name(0); print(torch.__version__, n); assert '4060' in n"],
    env=clean_env(), text=True, capture_output=True, check=True,
)
print(probe.stdout.strip())


## 2. 冻结 split 与训练配置

这里显式设置 `pretrained_path=null` 和 `resume=false`，所以不是从旧 80k 继续，而是从头学习。训练使用由 LeRobot 生成的 24-episode 连续重编号派生集，统计量也只由这 24 条聚合；原始 30 条 sanitized 数据保持不变，6 条验证不会进入训练或 normalization stats。

In [ ]:
IMAGE_TRANSFORMS = {
    'enable': True,
    'max_num_transforms': 1,
    'random_order': False,
    'tfs': {
        'identity': {'weight': 1.5, 'type': 'Identity', 'kwargs': {}},
        'brightness': {'weight': 1.0, 'type': 'ColorJitter', 'kwargs': {'brightness': [0.85, 1.15]}},
        'contrast': {'weight': 1.0, 'type': 'ColorJitter', 'kwargs': {'contrast': [0.85, 1.15]}},
        'saturation': {'weight': 1.0, 'type': 'ColorJitter', 'kwargs': {'saturation': [0.8, 1.2]}},
        'sharpness': {'weight': 1.0, 'type': 'SharpnessJitter', 'kwargs': {'sharpness': [0.8, 1.2]}},
        'affine': {
            'weight': 0.7,
            'type': 'RandomAffine',
            'kwargs': {'degrees': [-1.5, 1.5], 'translate': [0.01, 0.01], 'scale': [0.98, 1.02], 'interpolation': 2},
        },
        'erasing': {
            'weight': 0.7,
            'type': 'RandomErasing',
            'kwargs': {'p': 1.0, 'scale': [0.005, 0.03], 'ratio': [0.5, 2.0], 'value': 'random'},
        },
    },
}

def make_config(output_dir, dataset_root, repo_id, episodes, steps, save_freq, job_name):
    return {
        'dataset': {
            'repo_id': repo_id,
            'root': str(dataset_root),
            'episodes': episodes,
            'image_transforms': IMAGE_TRANSFORMS,
            'use_imagenet_stats': True,
            'video_backend': 'torchcodec',
        },
        'env': None,
        'policy': {
            'type': 'act',
            'pretrained_path': None,
            'chunk_size': 20,
            'n_action_steps': 1,
            'temporal_ensemble_coeff': None,
            'device': 'cuda',
            'use_amp': True,
            'push_to_hub': False,
        },
        'output_dir': str(output_dir),
        'job_name': job_name,
        'resume': False,
        'seed': 1000,
        'num_workers': 4,
        'batch_size': 8,
        'prefetch_factor': 4,
        'persistent_workers': True,
        'steps': steps,
        'eval_freq': 0,
        'log_freq': 1 if steps == 20 else 100,
        'save_checkpoint': True,
        'save_freq': save_freq,
        'use_policy_training_preset': True,
        'wandb': {'enable': False},
    }

smoke_config = make_config(SMOKE_RUN_DIR, TRAIN_DATASET_ROOT, TRAIN_REPO_ID, None, 20, 20, 'act_phase_c_aug_smoke')
full_config = make_config(FULL_RUN_DIR, TRAIN_DATASET_ROOT, TRAIN_REPO_ID, None, 80_000, 5_000, 'act_phase_c_aug_holdout')
SMOKE_CONFIG_PATH = ARTIFACT_DIR / 'smoke_config.json'
FULL_CONFIG_PATH = ARTIFACT_DIR / 'full_config.json'
SPLIT_PATH = ARTIFACT_DIR / 'split_manifest.json'
SMOKE_CONFIG_PATH.write_text(json.dumps(smoke_config, indent=2) + '\n')
FULL_CONFIG_PATH.write_text(json.dumps(full_config, indent=2) + '\n')
SPLIT_PATH.write_text(json.dumps({'train_episodes': TRAIN_EPISODES, 'val_episodes': VAL_EPISODES, 'heldout_rows': heldout_rows}, indent=2) + '\n')

schema_probe = """
import json, sys, draccus
from lerobot.configs.train import TrainPipelineConfig
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.transforms import ImageTransforms
payload = json.load(open(sys.argv[1]))
cfg = draccus.decode(TrainPipelineConfig, payload)
transform = ImageTransforms(cfg.dataset.image_transforms)
assert cfg.policy.pretrained_path is None and cfg.resume is False
assert cfg.dataset.episodes is None and cfg.policy.chunk_size == 20 and cfg.policy.n_action_steps == 1
print(cfg.policy.type, cfg.steps, cfg.dataset.episodes)
print(transform)
"""
subprocess.run([str(VENV_PYTHON), '-c', schema_probe, str(FULL_CONFIG_PATH)], env=clean_env(), check=True)
print('configs:', SMOKE_CONFIG_PATH, FULL_CONFIG_PATH)


## 3. 20-step smoke

只验证 CUDA、双视频解码、增强、前向/反向和 checkpoint 写入。

In [ ]:
def run_streaming(command, log_path):
    print('command:', ' '.join(map(str, command)))
    print('log:', log_path)
    with Path(log_path).open('w') as log_handle:
        process = subprocess.Popen(
            list(map(str, command)),
            cwd=WORKSPACE,
            env=clean_env(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'command failed with exit code {return_code}; see {log_path}')

run_streaming(
    [TRAIN_CLI, f'--config_path={SMOKE_CONFIG_PATH}'],
    ARTIFACT_DIR / 'smoke.log',
)
assert (SMOKE_RUN_DIR / 'checkpoints/000020/pretrained_model/model.safetensors').is_file()
print('smoke passed')


## 4. 从头训练 80k

成功条件不是训练 loss 继续下降，而是后面的 held-out checkpoint 指标优于较早 checkpoint，尤其闭爪阶段。

In [ ]:
run_streaming(
    [TRAIN_CLI, f'--config_path={FULL_CONFIG_PATH}'],
    ARTIFACT_DIR / 'train_80k.log',
)
assert (FULL_RUN_DIR / 'checkpoints/080000/pretrained_model/model.safetensors').is_file()
print('training complete:', FULL_RUN_DIR)


如果 notebook/主机在 80k 前中断，只运行下面这个 cell。它从数值最大的 checkpoint 恢复；完整跑完时会直接跳过。

In [ ]:
final_model = FULL_RUN_DIR / 'checkpoints/080000/pretrained_model/model.safetensors'
if final_model.is_file():
    print('80k already complete; resume skipped')
else:
    checkpoint_dirs = sorted(
        (path for path in (FULL_RUN_DIR / 'checkpoints').glob('[0-9]*') if path.name.isdigit()),
        key=lambda path: int(path.name),
    )
    if not checkpoint_dirs:
        raise FileNotFoundError('No checkpoint to resume. Run the full-training cell first.')
    resume_config = checkpoint_dirs[-1] / 'pretrained_model/train_config.json'
    run_streaming(
        [TRAIN_CLI, f'--config_path={resume_config}', '--resume=true', '--steps=80000', '--save_freq=5000', '--log_freq=100'],
        ARTIFACT_DIR / 'resume_to_80k.log',
    )


## 5. 对全部 checkpoint 做无增强 held-out 评估

排序分数是 `(body_mae_h1 + gripper_close_mae_h1) / 2`。它避免大量“夹爪保持打开”的帧把不会抓取的 checkpoint 伪装成好模型。

In [ ]:
METRICS_CSV = ARTIFACT_DIR / 'heldout_checkpoint_metrics.csv'
run_streaming(
    [
        VENV_PYTHON, EVALUATOR,
        '--run-dir', FULL_RUN_DIR,
        '--episodes', ','.join(map(str, VAL_EPISODES)),
        '--batch-size', '8',
        '--num-workers', '4',
        '--device', 'cuda',
        '--output-csv', METRICS_CSV,
    ],
    ARTIFACT_DIR / 'heldout_eval.log',
)

with METRICS_CSV.open(newline='') as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    for key in row:
        row[key] = int(row[key]) if key in {'step', 'samples', 'episodes'} else float(row[key])
ranked = sorted(rows, key=lambda row: row['selection_score'])
for row in ranked[:8]:
    print({key: round(row[key], 3) if isinstance(row[key], float) else row[key] for key in ('step', 'selection_score', 'mae_h1', 'mae_h5', 'mae_h20', 'body_mae_h1', 'gripper_close_mae_h1')})
BEST_STEP = ranked[0]['step']
BEST_CHECKPOINT = FULL_RUN_DIR / f'checkpoints/{BEST_STEP:06d}/pretrained_model'
print('offline candidate:', BEST_CHECKPOINT)
print('next gate: ARM LOCKED observation replay, then a separately authorized physical rollout')


## 6. 可选：用选出的步数在全部 30 条上重新拟合

先保留 24/6 模型做诊断和首次 locked replay。只有 held-out 指标确实改善时，才把 `RUN_REFIT` 改为 `True`；这会从头训练，不会覆盖 24/6 结果。

In [ ]:
RUN_REFIT = False
if RUN_REFIT:
    refit_name = f'{RUN_NAME}_all30_step{BEST_STEP:06d}'
    REFIT_RUN_DIR = BASE / 'outputs' / refit_name
    refit_config = make_config(REFIT_RUN_DIR, DATASET_ROOT, 'stevenzenith/hand_tracking_pv_carton_phase_b', None, BEST_STEP, 5_000, 'act_phase_c_aug_all30_refit')
    REFIT_CONFIG_PATH = ARTIFACT_DIR / 'refit_all30_config.json'
    REFIT_CONFIG_PATH.write_text(json.dumps(refit_config, indent=2) + '\n')
    run_streaming(
        [TRAIN_CLI, f'--config_path={REFIT_CONFIG_PATH}'],
        ARTIFACT_DIR / 'refit_all30.log',
    )
    print('all-30 refit:', REFIT_RUN_DIR)
else:
    print('refit disabled; review held-out metrics and locked replay first')
